In [ ]:
!pip install -q transformers torch tqdm h5py opencv-python-headless pillow

In [ ]:
import os, time, numpy as np, torch, torch.nn.functional as F, tarfile, cv2, h5py
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoProcessor

# Auto-detect and extract dataset
INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}
DATASET_BLACKLIST = {"my-hf-secrets", "kaggle-data-overlay"}

def find_and_extract_dataset():
    """Find dataset, extract tar if needed, return path with images"""
    if not INPUT_DIR.exists():
        return None
    for dataset_dir in sorted(INPUT_DIR.iterdir()):
        if not dataset_dir.is_dir():
            continue
        if dataset_dir.name in DATASET_BLACKLIST:
            continue
        tar_files = list(dataset_dir.glob("*.tar")) + list(dataset_dir.glob("*.tar.gz"))
        if tar_files:
            extract_dir = WORK_DIR / "dataset"
            extract_dir.mkdir(exist_ok=True)
            print(f"Extracting {tar_files[0].name}...")
            with tarfile.open(tar_files[0], 'r:*') as tar:
                tar.extractall(extract_dir)
            print(f"✓ Extracted to {extract_dir}")
            return str(extract_dir)
        for ext in SUPPORTED_EXTS:
            if list(dataset_dir.rglob(f"*{ext}")):
                return str(dataset_dir)
    return None

DATASET_PATH = find_and_extract_dataset()
if not DATASET_PATH:
    raise FileNotFoundError(f"No dataset found in {INPUT_DIR}")
print(f"✓ Dataset: {DATASET_PATH}")

MODEL_ID = "google/siglip2-base-patch16-naflex"
EMBEDDING_DIM = 768
OUTPUT_DIR = "/kaggle/working"

# Adaptive GPU/CPU configuration
# Primary target: T4 x2 (Kaggle free-tier)
# Fallback: P100, single GPU, CPU
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    if "P100" in gpu_name:
        # P100 (Pascal, CC 6.0) — fallback GPU, still uses CUDA
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 128, 4
        print(f"GPU: {gpu_name} (fallback) | Batch: {BATCH_SIZE}")
    elif num_gpus > 1:
        # T4 x2: ~224 images/GPU, well within 16GB VRAM each
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 448, 4
        print(f"GPU: {gpu_name} x{num_gpus} | Batch: {BATCH_SIZE}")
    else:
        device, BATCH_SIZE, NUM_WORKERS = "cuda", 256, 4
        print(f"GPU: {gpu_name} | Batch: {BATCH_SIZE}")
else:
    device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
    print("CPU mode")

In [ ]:
# SigLIP2 NaFlex supports both variable-resolution (aspect-ratio preserving)
# and fixed-resolution inputs. We force a FIXED target size to drop the
# expensive variable-shape collate_fn and let the DataLoader do a vanilla
# `torch.stack`. Tune NAFLEX_TARGET_SIDE to balance recall vs speed.
NAFLEX_TARGET_SIDE = 256  # multiple of 16 (patch size); 224/256/384 supported

class FastDataset(Dataset):
    """cv2 decode → resize → PIL → fixed-size AutoProcessor output."""
    def __init__(self, paths, proc, target_side):
        self.paths, self.proc, self.target_side = paths, proc, target_side
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = cv2.imread(self.paths[i], cv2.IMREAD_COLOR)
            if img is None:
                raise ValueError("cv2 read failed")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # Resize so the longest side == target_side, preserve aspect ratio
            h, w = img.shape[:2]
            if max(h, w) != self.target_side:
                scale = self.target_side / max(h, w)
                img = cv2.resize(img, (int(round(w * scale)), int(round(h * scale))),
                                 interpolation=cv2.INTER_AREA)
            img = Image.fromarray(img)
        except Exception:
            img = Image.new("RGB", (self.target_side, self.target_side), 0)
        inputs = self.proc(images=img, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in inputs.items()}, self.paths[i]


# Since every sample is now forced to the same aspect-preserving size, all
# batches have identical tensor shapes → vanilla `torch.stack` works.
def fixed_collate_fn(batch):
    """Standard collate — used because every image has the same shape."""
    input_dicts, paths = zip(*batch)
    collated = {k: torch.stack([d[k] for d in input_dicts], dim=0) for k in input_dicts[0]}
    return collated, list(paths)


# Scan for all images
imgs = [
    (str(p.relative_to(DATASET_PATH)), str(p))
    for p in Path(DATASET_PATH).rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
]
image_ids, image_paths = [i for i, _ in imgs], [p for _, p in imgs]

print(f"✓ Found {len(image_ids)} images")
if image_ids:
    print(f"  Sample: {image_ids[0]}")

In [ ]:
print(f"Loading {MODEL_ID}...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)

dtype = torch.float16 if device == "cuda" else torch.float32
use_amp = device == "cuda"
if use_amp: print("  Using FP16")

model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True)

# NOTE: do NOT delete text_model/text_projection —
# get_image_features() relies on internal model references
# that break if sub-modules are removed.
torch.cuda.empty_cache()

class VisionWrapper(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.model = m
    def forward(self, *args, **kwargs):
        out = self.model.get_image_features(*args, **kwargs)
        # Handle both direct Tensor return and BaseModelOutputWithPooling return
        if isinstance(out, torch.Tensor):
            return out
        return out.image_embeds if hasattr(out, 'image_embeds') else out.pooler_output

wrapped = VisionWrapper(model)

# torch.compile for single GPU only (incompatible with DataParallel)
if device == "cuda" and torch.cuda.device_count() == 1:
    try:
        # "reduce-overhead" uses CUDA graphs — biggest win for inference
        wrapped = torch.compile(wrapped, mode='reduce-overhead')
        print("  ✓ Applied torch.compile (reduce-overhead)")
    except Exception as e:
        print(f"  ⚠ torch.compile skipped: {e}")

if device == "cuda" and torch.cuda.device_count() > 1:
    final_model = torch.nn.DataParallel(wrapped).cuda()
    print("  ✓ DataParallel enabled")
else:
    final_model = wrapped.to(device)

final_model.eval()
if device == "cuda":
    torch.backends.cudnn.benchmark = True
    # Warmup pass so cuDNN picks the best kernels before timing starts
    with torch.no_grad():
        dummy = torch.randn(BATCH_SIZE, 3, NAFLEX_TARGET_SIDE, NAFLEX_TARGET_SIDE, device=device, dtype=dtype)
        for _ in range(2):
            _ = final_model(dummy)
        torch.cuda.synchronize()
    print("  ✓ cuDNN warmup done")
print(f"✓ Model ready ({time.time()-t0:.1f}s)")


In [ ]:
    loader = DataLoader(
        FastDataset(image_paths, processor, NAFLEX_TARGET_SIDE),
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda"),
        pin_memory_device="cuda" if device == "cuda" else "",
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=3 if NUM_WORKERS > 0 else None,
        # All samples share the same shape now → vanilla collate is enough.
        collate_fn=fixed_collate_fn,
    )

    # Defer H2D + numpy: keep fp16 features on GPU, do a single cat+transfer at the end.
    # Old code did .cpu().float().numpy() per batch — that forces a CUDA sync per batch
    # (~12 syncs for 5400 images / 448 batch). Removing those syncs is the biggest win.
    gpu_feats = []
    t0 = time.time()
    n_batches = 0

    with torch.inference_mode():  # stricter than no_grad (~5% faster)
        for inputs, _ in loader:
            inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}
            if use_amp:
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    feat = final_model(**inputs)
                    feat = F.normalize(feat, p=2, dim=-1)
            else:
                feat = final_model(**inputs)
                feat = F.normalize(feat, p=2, dim=-1)
            gpu_feats.append(feat)  # stay on GPU, no per-batch sync
            n_batches += 1

    # One sync + one cat + one transfer at the end
    torch.cuda.synchronize()
    sync_t = time.time() - t0
    embeddings = torch.cat(gpu_feats, dim=0).cpu().float().numpy()
    inf_time = time.time() - t0
    n_done = embeddings.shape[0]
    per_batch_ms = (inf_time * 1000 / n_batches) if n_batches else 0
    d2h_t = inf_time - sync_t
    print(f"✓ Inference: {inf_time:.1f}s ({n_done / inf_time:.1f} img/s, {n_batches} batches)")
    print(f"  Per-batch avg: {per_batch_ms:.0f}ms | cat+D2H at end: {d2h_t:.2f}s")



In [ ]:
# Save embeddings to HDF5
hdf5_path  = OUTPUT_DIR + "/embeddings.hdf5"

with h5py.File(hdf5_path, "w") as f:
    f.create_dataset("embeddings", data=embeddings)
    f.create_dataset("image_ids", data=[s.encode("utf-8") for s in image_ids])
    f.create_dataset("image_paths", data=[s.encode("utf-8") for s in image_paths])
    f.attrs["model_id"]      = MODEL_ID
    f.attrs["feature_type"]  = "image_features"
    f.attrs["embedding_dim"] = EMBEDDING_DIM
    f.attrs["num_images"]    = len(image_ids)

print(f"[OK] Saved embeddings.hdf5 ({embeddings.nbytes / 1024**2:.1f} MB)")

# Free memory (Kaggle session can be killed if RAM > 16GB)
del embeddings
import gc; gc.collect()
if device == "cuda": torch.cuda.empty_cache()
print("[OK] Pipeline complete")
